# SalesPilot — End-to-End Demo

This notebook demonstrates the SalesPilot CRM pipeline:
1. **Data Loading** — CSV ingestion & feature engineering
2. **Model Training** — XGBoost deal-closure predictor
3. **Scoring** — Predict win probability for accounts
4. **Route Optimization** — TSP-based visit route using OR-Tools
5. **API Test** — Hit the FastAPI endpoints

## 0. Setup

In [1]:
import sys, os
os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')
# Ensure we're in the salespilot project root
if os.path.basename(os.getcwd()) != 'salespilot':
    os.chdir('salespilot') if os.path.isdir('salespilot') else None
print(f'Working dir: {os.getcwd()}')
sys.path.insert(0, os.getcwd())

Working dir: /Users/kanakasarat/Downloads/CRM+Sales+Opportunities/salespilot


In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# CSV data is one level up
CSV_DIR = Path('..') 
print('CSV files available:', [f.name for f in CSV_DIR.glob('*.csv')])

CSV files available: ['products.csv', 'accounts.csv', 'sales_pipeline.csv', 'sales_teams.csv', 'data_dictionary.csv']


## 1. Data Loading & Exploration

In [3]:
# Load raw CSVs
accounts = pd.read_csv(CSV_DIR / 'accounts.csv')
products = pd.read_csv(CSV_DIR / 'products.csv')
sales_teams = pd.read_csv(CSV_DIR / 'sales_teams.csv')
pipeline = pd.read_csv(CSV_DIR / 'sales_pipeline.csv')

print(f'Accounts:    {accounts.shape}')
print(f'Products:    {products.shape}')
print(f'Sales Teams: {sales_teams.shape}')
print(f'Pipeline:    {pipeline.shape}')

Accounts:    (85, 7)
Products:    (7, 3)
Sales Teams: (35, 3)
Pipeline:    (8800, 8)


In [4]:
accounts.head()

,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,NaN
1,Betasoloin,medical,1999,251.41,495,United States,NaN
2,Betatech,medical,1986,647.18,1185,Kenya,NaN
3,Bioholding,medical,2012,587.34,1356,Philipines,NaN
4,Bioplex,medical,1991,326.82,1016,United States,NaN


In [5]:
pipeline.head()

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0


In [6]:
# Deal stage distribution
pipeline['deal_stage'].value_counts()

deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

## 2. Feature Engineering (data_loader pipeline)

In [7]:
from app.data.data_loader import _load_accounts, _load_products, _load_sales_teams, _load_opportunities, _hash_id

# Load and transform each table
acct_df = _load_accounts(CSV_DIR)
prod_df = _load_products(CSV_DIR)
team_df = _load_sales_teams(CSV_DIR)

print(f'Accounts (transformed): {acct_df.shape}')
print(f'Products (transformed): {prod_df.shape}')
print(f'Sales Teams (transformed): {team_df.shape}')
acct_df.head()

Accounts (transformed): (85, 10)
Products (transformed): (7, 4)
Sales Teams (transformed): (35, 4)


,account_id,account_name,industry,company_size,revenue,year_established,region,subsidiary_of,latitude,longitude
0,8836431947471538,Acme Corporation,technolgy,2822,1100.04,1996,United States,NaN,37.325009,-121.893420
1,5745241362957565,Betasoloin,medical,495,251.41,1999,United States,NaN,37.370005,-121.869358
2,5110844135380882,Betatech,medical,1185,647.18,1986,Kenya,NaN,33.483532,-112.057449
3,7143654564919261,Bioholding,medical,1356,587.34,2012,Philipines,NaN,36.151433,-115.131821
4,4253168845593269,Bioplex,medical,1016,326.82,1991,United States,NaN,37.291059,-121.935246


In [8]:
# Build FK lookups and load opportunities
account_lookup = dict(zip(
    acct_df['account_name'].str.strip().str.lower(),
    acct_df['account_id'],
))
agent_lookup = dict(zip(
    team_df['sales_agent'].str.strip().str.lower(),
    team_df['agent_id'],
))
product_lookup = dict(zip(
    prod_df['product_name'].str.strip().str.lower(),
    prod_df['product_id'],
))

opp_df = _load_opportunities(CSV_DIR, account_lookup, agent_lookup, product_lookup)
print(f'Opportunities (transformed): {opp_df.shape}')
opp_df.head()

Opportunities (transformed): (7375, 10)


,opportunity_id,account_id,agent_id,product_id,deal_value,sales_stage,engage_date,close_date,days_since_last_contact,deal_closed
0,3385693789861965,3231857028390948,4950957932798709,8800795766140891,1054.0,Won,2016-10-20,2017-03-01,3298,1
1,4288615006457101,1268475830787827,7971625204640674,<NA>,4514.0,Won,2016-10-25,2017-03-11,3288,1
2,4816390175053609,3231857028390948,7971625204640674,5099918575789089,50.0,Won,2016-10-25,2017-03-07,3292,1
3,7143232003420526,1728115136939395,4950957932798709,4471905713638513,588.0,Won,2016-10-25,2017-03-09,3290,1
4,3766527038648061,2318883451980450,4256506655091195,4471905713638513,517.0,Won,2016-10-25,2017-03-02,3297,1


In [9]:
# Check deal_closed distribution
print('Deal closed distribution:')
print(opp_df['deal_closed'].value_counts())
print(f'\nWin rate: {opp_df["deal_closed"].mean():.2%}')

Deal closed distribution:
deal_closed
1    4238
0    3137
Name: count, dtype: int64

Win rate: 57.46%


## 3. Synthetic Geo Coordinates

In [10]:
print(f'Accounts with coordinates: {acct_df["latitude"].notna().sum()} / {len(acct_df)}')
print(f'\nSample coordinates:')
acct_df[['account_name', 'region', 'latitude', 'longitude']].head(10)

Accounts with coordinates: 85 / 85

Sample coordinates:


,account_name,region,latitude,longitude
0,Acme Corporation,United States,37.325009,-121.893420
1,Betasoloin,United States,37.370005,-121.869358
2,Betatech,Kenya,33.483532,-112.057449
3,Bioholding,Philipines,36.151433,-115.131821
4,Bioplex,United States,37.291059,-121.935246
5,Blackzim,United States,37.369444,-121.918372
6,Bluth Company,United States,37.373508,-121.874269
7,Bubba Gump,United States,37.349833,-121.880933
8,Cancity,United States,37.365354,-121.839669
9,Cheers,United States,37.324996,-121.868918


## 4. Model Training

In [11]:
from app.ml.train_model import load_features_from_csv, train, build_preprocessor

# Load features from CSV (no DB needed)
feature_df = load_features_from_csv(str(CSV_DIR))
print(f'Feature DataFrame: {feature_df.shape}')
print(f'Class balance: {feature_df["deal_closed"].value_counts().to_dict()}')
feature_df.head()

Feature DataFrame: (7375, 13)
Class balance: {1: 4238, 0: 3137}


,opportunity_id,sales_agent,product,account_name,sales_stage,engage_date,close_date,deal_value,deal_closed,days_since_last_contact,industry,company_size,region
0,1C1I7A6R,Moses Frase,GTX Plus Basic,Cancity,Won,2016-10-20,2017-03-01,1054.0,1,3298,retail,2448.0,United States
1,Z063OYW0,Darcel Schlecht,GTXPro,Isdom,Won,2016-10-25,2017-03-11,4514.0,1,3288,medical,4540.0,United States
2,EC4QE1BX,Darcel Schlecht,MG Special,Cancity,Won,2016-10-25,2017-03-07,50.0,1,3292,retail,2448.0,United States
3,MV1LWRNH,Moses Frase,GTX Basic,Codehow,Won,2016-10-25,2017-03-09,588.0,1,3290,software,2641.0,United States
4,PE84CX4O,Zane Levy,GTX Basic,Hatfan,Won,2016-10-25,2017-03-02,517.0,1,3297,services,1299.0,United States


In [12]:
# Train the model
metrics = train(feature_df)
print('\nTraining metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

Model saved to /Users/kanakasarat/Downloads/CRM+Sales+Opportunities/salespilot/app/ml/artifacts/model.joblib
Metrics: {
  "train_auc": 1.0,
  "train_accuracy": 1.0,
  "val_auc": 1.0,
  "val_accuracy": 1.0,
  "test_auc": 1.0,
  "test_accuracy": 1.0
}

Training metrics:
  train_auc: 1.0
  train_accuracy: 1.0
  val_auc: 1.0
  val_accuracy: 1.0
  test_auc: 1.0
  test_accuracy: 1.0


## 5. Model Scoring (Direct — No DB)

In [13]:
import joblib

# Load the trained model
model = joblib.load('app/ml/artifacts/model.joblib')

# Build a sample scoring DataFrame
sample_df = feature_df.sample(10, random_state=42)[[
    'industry', 'region', 'sales_stage', 'deal_value',
    'company_size', 'days_since_last_contact'
]].copy()

# Score
probs = model.predict_proba(sample_df)[:, 1]
sample_df['win_probability'] = probs
sample_df.sort_values('win_probability', ascending=False)

,industry,region,sales_stage,deal_value,company_size,days_since_last_contact,win_probability
2186,services,United States,Won,1091.0,5276.0,3259,0.999457
4507,finance,United States,Won,4565.0,5113.0,3136,0.999457
8180,entertainment,United States,Won,953.0,978.0,3009,0.999455
2937,finance,United States,Won,1101.0,883.0,3232,0.999455
2018,retail,United States,Won,976.0,7523.0,3260,0.999455
1185,technolgy,United States,Won,51.0,1806.0,3210,0.999454
6177,marketing,United States,Engaging,0.0,100.0,3121,0.000629
2918,medical,United States,Lost,0.0,6837.0,3228,0.000540
7213,finance,United States,Lost,0.0,7227.0,3080,0.000540
7820,marketing,Poland,Lost,0.0,1593.0,3050,0.000540


## 6. Route Optimization (TSP)

In [14]:
from app.optimization.haversine import haversine_km, build_distance_matrix

# Pick 6 accounts with coordinates as sample stops
sample_accounts = acct_df.head(6)
points = list(zip(sample_accounts['latitude'], sample_accounts['longitude']))

print('Sample account stops:')
for i, (_, row) in enumerate(sample_accounts.iterrows()):
    print(f'  [{i}] {row["account_name"]} — ({row["latitude"]:.4f}, {row["longitude"]:.4f})')

# Build distance matrix
dist_matrix = build_distance_matrix(points)
print(f'\nDistance matrix (km):')
print(np.round(dist_matrix, 1))

Sample account stops:
  [0] Acme Corporation — (37.3250, -121.8934)
  [1] Betasoloin — (37.3700, -121.8694)
  [2] Betatech — (33.4835, -112.0574)
  [3] Bioholding — (36.1514, -115.1318)
  [4] Bioplex — (37.2911, -121.9352)
  [5] Blackzim — (37.3694, -121.9184)

Distance matrix (km):
[[  0.    5.4 987.8 616.3   5.3   5.4]
 [  5.4   0.  987.8 615.2  10.5   4.3]
 [987.8 987.8   0.  408.3 989.8 991.8]
 [616.3 615.2 408.3   0.  619.3 619.4]
 [  5.3  10.5 989.8 619.3   0.    8.8]
 [  5.4   4.3 991.8 619.4   8.8   0. ]]


In [15]:
# Run TSP solver in a subprocess to avoid protobuf conflict between ortools and pandas/pyarrow
import subprocess, json as _json

points_json = _json.dumps(points)
names = sample_accounts['account_name'].tolist()

script = f'''
import sys, json
sys.path.insert(0, ".")
from app.optimization.distance_provider import HaversineProvider
from app.optimization.ortools_tsp import solve_tsp

points = json.loads('{points_json}')
points = [tuple(p) for p in points]
provider = HaversineProvider()
result = solve_tsp(points, provider)
output = json.dumps(dict(ordered_indices=result.ordered_indices, total_distance_km=result.total_distance_km))
print(output)
'''

proc = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True, timeout=30)
if proc.returncode != 0:
    print('TSP solver error:', proc.stderr)
else:
    tsp_result = _json.loads(proc.stdout.strip())
    ordered = tsp_result['ordered_indices']
    dist = tsp_result['total_distance_km']
    print('Optimal route order:', ordered)
    print('Total distance:', dist, 'km')
    print('\nRoute:')
    for i, idx in enumerate(ordered):
        print(f'  Stop {i}: {names[idx]}')
    print(f'  Return to: {names[ordered[0]]}')

Optimal route order: [0, 4, 2, 3, 1, 5]
Total distance: 2028.3 km

Route:
  Stop 0: Acme Corporation
  Stop 1: Bioplex
  Stop 2: Betatech
  Stop 3: Bioholding
  Stop 4: Betasoloin
  Stop 5: Blackzim
  Return to: Acme Corporation


## 7. API Test (FastAPI)

In [16]:
import httpx

BASE_URL = 'http://localhost:8000'

# Health check
try:
    r = httpx.get(f'{BASE_URL}/health', timeout=5)
    print(f'Health: {r.json()}')
except httpx.ConnectError:
    print('API not running — start with: uvicorn app.main:app --port 8000')

Health: {'status': 'ok'}


In [17]:
# OpenAPI spec summary
try:
    r = httpx.get(f'{BASE_URL}/openapi.json', timeout=5)
    spec = r.json()
    print('API Endpoints:')
    for path, methods in spec['paths'].items():
        for method in methods:
            print(f'  {method.upper()} {path}')
except httpx.ConnectError:
    print('API not running')

API Endpoints:
  GET /health
  POST /v1/load-data
  POST /v1/score-accounts
  POST /v1/optimize-route


## 8. Summary

| Component | Status |
|---|---|
| CSV Data Loading (4 tables) | Tested |
| Feature Engineering | Tested |
| Synthetic Geo Coordinates | Tested |
| XGBoost Model Training | Tested |
| Model Scoring (predict_proba) | Tested |
| Haversine Distance Matrix | Tested |
| OR-Tools TSP Solver | Tested |
| FastAPI Health Endpoint | Tested |